# 感知机（Perceptron）——从零实现

**目标**：用 20 行 Python，不依赖任何机器学习库，从零实现一个能自己「学习」的感知机。

只需要：`numpy`（做数学运算）和 `matplotlib`（画图）。

跑完这个 notebook，你会看到感知机**一步步调整，直到完全答对**。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

# 固定随机种子，保证每次跑结果一样
np.random.seed(42)

## 第一步：造数据

我们造两类点：
- **蓝点**（标签 = 0）：x₁ 和 x₂ 都比较小
- **红点**（标签 = 1）：x₁ 和 x₂ 都比较大

这两类点能用一条直线分开——感知机就能学会区分它们。

In [ ]:
# 造两类点，每类 20 个
n = 20

# 蓝点：围绕 (1, 1) 随机分布
blue = np.random.randn(n, 2) * 0.5 + np.array([1.0, 1.0])
# 红点：围绕 (3, 3) 随机分布
red  = np.random.randn(n, 2) * 0.5 + np.array([3.0, 3.0])

# 合并数据
X = np.vstack([blue, red])      # 形状 (40, 2)：40 个点，每个有 2 个特征
y = np.array([0]*n + [1]*n)     # 前 20 个是 0（蓝），后 20 个是 1（红）

# 画出来看看
plt.figure(figsize=(5, 5))
plt.scatter(blue[:, 0], blue[:, 1], color='steelblue', label='标签 0（蓝）', s=60)
plt.scatter(red[:, 0],  red[:, 1],  color='crimson',   label='标签 1（红）', s=60)
plt.title('数据集（两类，线性可分）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('data.png', dpi=80, bbox_inches='tight')
plt.show()
print(f'数据集大小：{len(X)} 个点，{n} 蓝 + {n} 红')

## 第二步：实现感知机

感知机做三件事：

1. **predict（预测）**：拿权重，算加权求和，超过阈值就输出 1，否则 0
2. **train（训练）**：对每个例子，答错了就调整权重
3. 重复 train 多轮，直到全部答对（或达到最大轮数）

In [ ]:
class Perceptron:
    """从零实现感知机。不用任何机器学习库。"""

    def __init__(self, n_features, learning_rate=0.1):
        # 权重初始化为全 0（也可以随机，效果一样）
        self.w = np.zeros(n_features)
        self.bias = 0.0           # 偏置，相当于阈值的反面
        self.lr = learning_rate
        self.history = []         # 记录每轮错误数

    def predict(self, x):
        """给一个输入 x，返回 0 或 1。"""
        # 加权求和 + 偏置
        weighted_sum = np.dot(x, self.w) + self.bias
        # 超过 0 就输出 1，否则输出 0
        return 1 if weighted_sum > 0 else 0

    def train_one_epoch(self, X, y):
        """对所有数据跑一轮，更新权重。返回本轮错误数。"""
        errors = 0
        for xi, yi in zip(X, y):
            y_hat = self.predict(xi)   # 我猜的答案
            if y_hat != yi:            # 猜错了
                delta = self.lr * (yi - y_hat)
                self.w    += delta * xi   # 调整权重
                self.bias += delta        # 调整偏置
                errors += 1
        return errors

    def fit(self, X, y, max_epochs=100):
        """训练到收敛或达到最大轮数。供测试和快捷调用使用。"""
        self.history = []
        for epoch in range(max_epochs):
            errors = self.train_one_epoch(X, y)
            self.history.append(errors)
            if errors == 0:
                return self
        return self

    def accuracy(self, X, y):
        """计算在数据集上的准确率。"""
        correct = sum(self.predict(xi) == yi for xi, yi in zip(X, y))
        return correct / len(y)

# 注：此实现与 src/perceptron.py 完全一致，pytest tests/ 测试的是同一份代码。
# fit() 是便捷包装；Cell 7 用手动 for 循环展示训练过程内部机制。


## 第三步：训练，看权重如何变化

In [ ]:
p = Perceptron(n_features=2, learning_rate=0.1)

print('轮次 | 错误数 | 准确率')
print('-----|--------|-------')

history = []  # 记录每轮准确率，用于画图
max_epochs = 30

for epoch in range(1, max_epochs + 1):
    errors = p.train_one_epoch(X, y)
    acc = p.accuracy(X, y)
    history.append(acc)
    print(f'  {epoch:2d}  |   {errors:2d}   |  {acc:.0%}')
    if errors == 0:
        print(f'\n✓ 第 {epoch} 轮达到 100% — 感知机收敛了！')
        break

## 第四步：画学习曲线 + 决策边界

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# --- 左图：学习曲线 ---
ax1.plot(range(1, len(history)+1), [a*100 for a in history],
         marker='o', color='steelblue', linewidth=2)
ax1.axhline(100, color='green', linestyle='--', alpha=0.5, label='100%')
ax1.set_xlabel('训练轮次')
ax1.set_ylabel('准确率 (%)')
ax1.set_title('学习曲线：准确率随训练轮次上升')
ax1.set_ylim(0, 110)
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- 右图：决策边界 ---
ax2.scatter(blue[:, 0], blue[:, 1], color='steelblue', label='标签 0（蓝）', s=60, zorder=3)
ax2.scatter(red[:, 0],  red[:, 1],  color='crimson',   label='标签 1（红）', s=60, zorder=3)

# 决策边界：w[0]*x + w[1]*y + bias = 0  →  y = -(w[0]*x + bias) / w[1]
if abs(p.w[1]) > 1e-9:
    x_range = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200)
    y_boundary = -(p.w[0] * x_range + p.bias) / p.w[1]
    ax2.plot(x_range, y_boundary, 'k-', linewidth=2, label='决策边界')

ax2.set_title('训练结束后的决策边界')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('result.png', dpi=80, bbox_inches='tight')
plt.show()
print(f'最终权重: w = {p.w}, bias = {p.bias:.2f}')

## 第五步：感知机解不了 XOR

现在试一个感知机永远学不会的问题——XOR（异或）。

| x₁ | x₂ | 正确答案 |
|----|----|---------|
|  0 |  0 |    0    |
|  0 |  1 |    1    |
|  1 |  0 |    1    |
|  1 |  1 |    0    |

你能用一条直线把「答案为 1」的点和「答案为 0」的点分开吗？试试看——

In [ ]:
# XOR 数据集
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

# 训练感知机
p_xor = Perceptron(n_features=2, learning_rate=0.1)
print('在 XOR 数据上训练 100 轮：')
for epoch in range(100):
    errors = p_xor.train_one_epoch(X_xor, y_xor)

final_acc = p_xor.accuracy(X_xor, y_xor)
print(f'\n训练 100 轮后，准确率：{final_acc:.0%}')
print('（永远达不到 100%——这就是感知机的极限）')

# 画出来
fig, ax = plt.subplots(figsize=(5, 5))
colors = ['steelblue' if yi == 0 else 'crimson' for yi in y_xor]
labels_shown = ['标签 0', '标签 0', '标签 1', '标签 1']
for i, (xi, yi, c) in enumerate(zip(X_xor, y_xor, colors)):
    ax.scatter(xi[0], xi[1], color=c, s=200, zorder=3)
    ax.annotate(f'  ({xi[0]:.0f},{xi[1]:.0f})→{yi}', xi, fontsize=11)

if abs(p_xor.w[1]) > 1e-9:
    x_range = np.linspace(-0.5, 1.5, 200)
    y_boundary = -(p_xor.w[0] * x_range + p_xor.bias) / p_xor.w[1]
    ax.plot(x_range, y_boundary, 'k-', linewidth=2, label='感知机找到的"边界"')

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_title('XOR 问题——没有一条直线能分开')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('xor.png', dpi=80, bbox_inches='tight')
plt.show()
print('\n→ 这个问题引出了下一个节点：多层感知机（1969-1986）')

## 总结

| | |
|---|---|
| **感知机能做** | 把线性可分的两类数据分开 |
| **感知机怎么学** | 答错了就调权重，重复直到全对 |
| **感知机做不到** | XOR——这条直线画不出来 |
| **历史意义** | 1958 年证明：机器可以从例子中自动学习 |
| **引出下一节** | XOR 问题→感知机局限→AI 寒冬→多层网络 |

**核心代码只有 15 行**（`predict` + `train_one_epoch`）——这就是机器学习史上第一个学习算法。

---
*引用*：Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. *Psychological Review*, 65(6), 386–408. https://doi.org/10.1037/h0042519